# Actividad 3
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2-2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad3_Sakuragi_Mitsui_Rukagua_Sendoh.ipynb
* Subir el archivo al link de entrega Actividad 3 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 20 de septiembre de 2026 - 23:59 horas chile.

__Integrantes:__ (RUT, Nombre y Apellido)

* 
* 
* 

## Librerias

In [ ]:
!python -m pip install opencv-python scikit-image

In [ ]:
## Misc
import os
from glob import glob
from pandas import DataFrame

## Visualization
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
%matplotlib inline

## Pre-processing
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Model
from tensorflow.keras import Model, layers

## Agregar las otras librerias que necesiten


## Funciones personalizadas

#### Graficador de desempeño

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model

  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

#### [Dice Similarity Coefficient (DSC)](https://en.wikipedia.org/wiki/S%C3%B8rensen%E2%80%93Dice_coefficient)

In [ ]:
def dice_similarity_coef(y_true, y_pred, smooth=100):
    """
        DESCRIPTION:
            Compute Dice Similarity Coefficient (DSC),
            the higher the better.

        INPUT:
            @param y_true: true image
            @type y_true: tensorflow.tensor

            @param y_pred: predicted image
            @type y_pred: tensorflow.tensor

            @param smooth: smooth-value to prevent (c/0)-error (default 100)
            @type smooth: float

        OUTPUT:
            @param DSC: Dice Similarity Coefficient
            @type DSC: tensorflow.tensor

    """

    y_truef = K.flatten(y_true)
    y_predf = K.flatten(y_pred)

    ## Compute intersection cardinality
    card_intersection = K.sum(y_truef * y_predf)

    ## Compute DSC
    DSC = (2* card_intersection + smooth) / (K.sum(y_truef) + K.sum(y_predf) + smooth)

    ## Return Dice Similarity Coefficient
    return DSC


def dice_coef_loss(y_true, y_pred):
    """
    DESCRIPTION:
        Compute Dice Similarity Coefficient loss

    INPUT:
        @param y_true: true image
        @type y_true: tensorflow.tensor

        @param y_pred: predicted image
        @type y_pred: tensorflow.tensor

    OUTPUT:
        @param loss: Dice Similarity Coefficient loss
        @type loss: tensorflow.tensor

    """
    ## Compute DSC loss
    loss = -dice_similarity_coef(y_true, y_pred)

    ## return loss
    return loss

#### [Intersection over union (IoU)](https://hasty.ai/content-hub/mp-wiki/metrics/iou-intersection-over-union)

In [ ]:
def iou(y_true, y_pred, smooth=100):
    """
        DESCRIPTION:
            Compute Intersection over Union (IoU), or Jaccard index

        INPUT:
            @param y_true: true image
            @type y_true: tensorflow.tensor

            @param y_pred: predicted image
            @type y_pred: tensorflow.tensor

        OUTPUT:
            @param iou_value: Intersection over Union
            @type iou_value: tensorflow.tensor

    """
    y_truef = K.flatten(y_true)
    y_predf = K.flatten(y_pred)

    ## Intersection area
    intersection = K.sum(y_true * y_pred)

    ## sum both areas
    sum_areas = K.sum(y_true + y_pred)

    ## Compute IoU
    iou_value = (intersection + smooth) / (sum_areas - intersection + smooth)

    ## return IoU value
    return iou_value

def iou_loss(y_true, y_pred):
    """
        DESCRIPTION:
            Compute Intersection over Union (IoU) loss

        INPUT:
            @param y_true: true image
            @type y_true: tensorflow.tensor

            @param y_pred: predicted image
            @type y_pred: tensorflow.tensor

        OUTPUT:
            @param loss: Intersection over Union (IoU) loss
            @type loss: tensorflow.tensor

    """
    ## Compute IoU loss
    loss = -iou(y_true, y_pred)

    ## return loss
    return loss

#### Data generator

In [ ]:
def normalizer(img, mask):
    """
        DESCRIPTION:
            Image normalizer, scaling valur into [0,1]-values

        INPUT:
            @param img: raw image
            @type img: numpy.ndarray

            @param mask: mask image
            @type mask: numpy.ndarray

        OUTPUT:
            @param batch_sample: raw and mask images tuple.
            @type batch_sample: tuple.
    """

    ## Normalizing
    img = img / 255
    mask = mask / 255
    mask[mask > 0.5] = 1
    mask[mask <= 0.5] = 0

    ## Batch samples
    batch_sample = (img, mask)

    ## Return batch images
    return batch_sample

def image_generator(data_frame,
                    batch_size,
                    augmentation_setting = {},
                    image_color_mode='rgb',
                    mask_color_mode='grayscale',
                    image_save_prefix='image',
                    mask_save_prefix='mask',
                    save_to_dir=None,
                    target_size=(256,256),
                    random_seed=1
                    ):
    """
        DESCRIPTION:
            can generate image and mask at the same time use the same seed for
            image_datagen and mask_datagen to ensure the transformation for image
            and mask is the same if you want to visualize the results of generator,
            set save_to_dir = "your path"

        INPUT:
            @param data_frame: raw and mask images paths
            @type data_frame: pandas.DataFrame

            @param batch_size: batch size of images
            @type batch_size: int

            @param augmentation_setting: image generator augmentation's setting
            @type augmentation_setting: dict

            @param image_color_mode: raw image color mode (default rgb)
            @type image_color_mode: str

            @param mask_color_mode: mask image color mode (default grayscale)
            @type mask_color_mode: str

            @param image_save_prefix: raw image filename prefix (default image)
            @type image_save_prefix: str

            @param mask_save_prefix: mask image filename prefix (default mask)
            @type mask_save_prefix: str

            @param save_to_dir: storage path (default None)
            @type save_to_dir: str or None

            @param target_size: image resizing (default (256,256))
            @type target_size: tuple

            @param random_seed: random seed value
            @type random_seed: int

        OUTPUT:
            @param batch_sample: raw and mask images batch tuple
            @type batch_sample: tuple
    """

    ## Image generator instances
    image_datagen = ImageDataGenerator(**augmentation_setting)
    mask_datagen = ImageDataGenerator(**augmentation_setting)

    ## raw image generator
    image_generator = image_datagen.flow_from_dataframe(
                          data_frame,
                          x_col = 'image_path',
                          class_mode = None,
                          color_mode = image_color_mode,
                          target_size = target_size,
                          batch_size = batch_size,
                          save_to_dir = save_to_dir,
                          save_prefix  = image_save_prefix,
                          seed = random_seed)

    ## mask image generator
    mask_generator = mask_datagen.flow_from_dataframe(
                          data_frame,
                          x_col = 'mask_path',
                          class_mode = None,
                          color_mode = mask_color_mode,
                          target_size = target_size,
                          batch_size = batch_size,
                          save_to_dir = save_to_dir,
                          save_prefix  = mask_save_prefix,
                          seed = random_seed)

    ## samples generator
    sample_generator = zip(image_generator, mask_generator)

    for (img, mask) in sample_generator:

        ## Image normalizing
        img, mask = normalizer(img, mask)

        ## Batching
        batch_sample = (img,mask)

        ## Return batch samples
        yield batch_sample

## Problem: Brain MRI Segmentation

__Target__: Mask detection

<center>
    <img src=https://proyecto-grupo-1-segmentacion-de-tumores-cerebra-07173d18d3f274.pages.fing.edu.uy/assets/img/img6.png width=800>
</center>

## Dataset

Source: [Brain MRI Segmentation](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation) (Kaggle) <br>
Alternative source: [MRI_Images.zip](https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/acholo_alumnos_uai_cl/ESTVM4NmZD5HiFKWXlMzNs4B2ADtDtCcXj4Fqw4v_0gCVA?e=ThLZ8q&download=1)

In [ ]:
## Solo para plataforma Linux o MacOS
if os.name == 'posix':
    if not os.path.exists("MRI_Images.zip"):

        ## Download file and rename
        !wget "https://alumnosuaicl-my.sharepoint.com/:u:/g/personal/acholo_alumnos_uai_cl/ESTVM4NmZD5HiFKWXlMzNs4B2ADtDtCcXj4Fqw4v_0gCVA?e=ThLZ8q&download=1" -O "MRI_Images.zip"

        ## Un-compress downloaded zip file
        !unzip MRI_Images.zip

    elif not os.path.exists("MRI_Images"):

        ## Un-compress downloaded zip file
        !unzip MRI_Images.zip

In [ ]:
## Image resizing
IMG_SIZE = 512

## Load filenames
raw_files = []

## Mask filenames
mask_files = glob('MRI_Images/*/*_mask*')
mask_files.sort()

## Raw filenames
for i in mask_files:
    raw_files.append(i.replace('_mask',''))

## Display top k.
k = 5
for name in zip(raw_files[:k], mask_files[:k]):
  print(name[0], " --- ", name[1])

In [ ]:
## Patience id
if os.name == 'posix':
    patiences = [name.split('/')[1] for name in raw_files]
else:
    patiences = [name.split('\\')[1] for name in raw_files]

## Patience - raw filename - mask filename
df = DataFrame({'patient': patiences,
                'image_path': raw_files,
                'mask_path': mask_files})

## Display number of samples
print('Number of samples:', len(df))

df.head(5)

#### Pre-procesamiento

In [ ]:
## Data partition
df_train, df_test = train_test_split(df, test_size = 0.1,
                                     random_state = 84)
df_train, df_val = train_test_split(df_train, test_size = 0.1,
                                    random_state = 84)

## Data shape display
print('(Shape) train: {}'.format(df_train.values.shape))
print('(Shape) validate: {}'.format(df_val.values.shape))
print('(Shape) test: {}\n'.format(df_test.values.shape))


## Pregunta 1 (5 pts):

Usando el código vista en clase, entrene un modelo LadderNet de 2 etapas con 15 filtros iniciales, una profundida de 4 y una cantidad de épocas que evite sobreajuste. Entregue el valor de la métrica IoU y DICE del conjunto de validación.  

## Pregunta 2 (5 pts):

Usando el código vista en clase, entrene un modelo LadderNet de 3 etapas con una cantidad de filtros iniciales y profundidad que usted elijan y una cantidad de épocas que evite sobreajuste con el objetivo que puedan maximizar el IoU en el conjunto de validación. Entregue el valor de la métrica IoU y DICE del conjunto de validación.  

## Pregunta 3 (2 pts):

Analicen y comparen los resultados obtenidos en las preguntas 1 y 2. Indique ¿Cuál de los dos modelos se quedarían? justifiquen. 

## Pregunta 4 (3 pts):

Del modelo seleccionado en la pregunta 3, muestre 10 predicciones del conjunto de test con su respectivas imagenes reales y de MRI's.   